# Full-data pipeline run: EDA + blocking → training artifacts

One notebook that runs everything upstream of the matcher on the **entire** dataset and leaves
its results on disk for the training pipeline:

1. executes `01_nlp_cleaning_eda.ipynb` and `02_blocking.ipynb` with full-data parameters (via
   [papermill](https://papermill.readthedocs.io)) and saves the executed copies with their outputs;
2. cleans every source file and builds the blocking keys (reused if 02 already did it);
3. generates candidates for **all 2.2M train** S1 entities, labelled against the ground truth: the
   matcher's training set;
4. generates candidates for **all 1.7M test** S1 entities → `output/candidate_pairs.tsv` (validated) plus
   a feature-ready copy;
5. writes a metrics/config report.

### Artifacts
| path | contents | used for |
|---|---|---|
| `cleaned/{train,test}_source{1,2,3}.parquet` | raw + cleaned columns per record (`name_norm`, `name_core`, `legal_form`, `addr_norm`, `region`, `postcode`, `house_no`) | matcher features |
| `artifacts/candidates_train/part-*.parquet` | one row per (S1, candidate): `s1_entity_id`, `cand_entity_id`, `score`, `passes`, `score_{all,name,addr}`, `rank_{all,name,addr}`, **`label`** | matcher training data |
| `artifacts/candidates_test/part-*.parquet` | same, without `label` | matcher inference |
| `artifacts/blocking_per_s1_{train,test}.parquet` | per S1: candidates, true matches, true matches found | error analysis, singleton handling |
| `artifacts/blocking_report.json` | blocking metrics on full train/test + the exact pass config | documentation |
| `output/candidate_pairs.tsv` | submission-format candidate list for test | final submission |
| `notebooks/executed/*.ipynb` | executed copies of notebooks 01 and 02 | review |

All of `cleaned/`, `blocking_cache/` and `artifacts/` are gitignored (several GB).

### Resources (rough)
* **Disk**: ~3 GB cleaned parquet + ~4 GB key cache + ~2 GB candidates.
* **RAM**: blocking is batched and runs in ~3 GB. Notebook 01 on the *entire* dataset loads all 22M
  raw records into pandas: plan for **32 GB+**, or lower `EDA_PARAMS["SAMPLE_ROWS"]` below (its default
  of 300k rows per file gives the same plots and rates).
* **Time** (8-core laptop, extrapolated from a 250k-record slice): cleaning ~15–25 min and key building
  ~10–15 min (once); train candidates ~1.5 h, test candidates ~1 h; notebook 02's experiments ~30–60 min.
  Retrieval is multi-threaded (`sparse_dot_topn`), so more cores help directly.

Every step is idempotent where it can be: cleaned files and key caches are reused if present.
Set the `RUN_*` flags to skip steps.

In [ ]:
RUN_EDA = True                 # execute notebook 01
RUN_BLOCKING_TUNING = True     # execute notebook 02 (experiments on a train validation sample)
RUN_TRAIN_CANDIDATES = True
RUN_TEST_CANDIDATES = True

# Notebook 01 on the entire dataset: SAMPLE_ROWS >= file size keeps every row, N_GT_ENTITIES = all S1.
# Low-RAM alternative: SAMPLE_ROWS=300_000, N_GT_ENTITIES=20_000 (the notebook's defaults).
EDA_PARAMS = dict(SAMPLE_ROWS=10**9, N_GT_ENTITIES=2_206_821, CHUNK=500_000, RUN_FULL_CLEAN=False)
# Notebook 02 always searches the full pool; EVAL_N is the size of its tuning sample. The full-train
# evaluation of the final configuration is done below, on all 2.2M entities.
BLOCKING_PARAMS = dict(JOBS=2, EVAL_N=100_000, RUN_TEST=False)

JOBS = 2             # processes for cleaning / key building (~1 GB RAM each)
BATCH_CHUNKS = 2     # S1 key chunks (250k records each) per candidate-generation batch; lower if RAM is tight

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for base in [start, *start.parents]:
        if (base / "code/business_entity_resolution/src/blocking.py").exists():
            return base
    raise FileNotFoundError("run this notebook from inside the repository")


ROOT = find_repo_root()
SRC = str(ROOT / "code/business_entity_resolution/src")
sys.path.insert(0, SRC)
# worker processes (cleaning / key building) start fresh interpreters on Windows/macOS: pass the path on
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [SRC, os.environ.get("PYTHONPATH")]))
import blocking as B   # noqa: E402
import cleaning as C   # noqa: E402

DATA_DIR = next(p for p in ROOT.glob("**/student_resource/dataset") if "__MACOSX" not in p.parts)
NB_DIR = ROOT / "notebooks"
EXECUTED_DIR = NB_DIR / "executed"
CLEAN_DIR = ROOT / "cleaned"
CACHE_DIR = ROOT / "blocking_cache"
ARTIFACT_DIR = ROOT / "artifacts"
OUT_DIR = ROOT / "output"
for d in (EXECUTED_DIR, ARTIFACT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

timings = {}


def timed(name):
    """Context manager recording wall time per step in `timings`."""
    class _T:
        def __enter__(self):
            self.t0 = time.time()

        def __exit__(self, *exc):
            timings[name] = round(time.time() - self.t0)
            print(f"[{name}] {timings[name]}s")
    return _T()


print(ROOT, DATA_DIR, sep="\n")

## 1. Execute notebooks 01 and 02
Each runs in its own kernel with the parameters above injected after its `parameters` cell. The
executed copies (with all outputs) land in `notebooks/executed/`; the source notebooks are untouched.
A failure stops here and the partially executed copy shows the failing cell.

In [ ]:
import papermill as pm  # noqa: E402

runs = [("01_nlp_cleaning_eda", RUN_EDA, EDA_PARAMS), ("02_blocking", RUN_BLOCKING_TUNING, BLOCKING_PARAMS)]
for name, enabled, params in runs:
    if not enabled:
        print(f"skipping {name}")
        continue
    with timed(f"notebook {name}"):
        pm.execute_notebook(NB_DIR / f"{name}.ipynb", EXECUTED_DIR / f"{name}.full.ipynb",
                            parameters=params, cwd=str(NB_DIR), log_output=False, progress_bar=True)

## 2. Clean all files and build key caches
No-ops when notebook 02 (or an earlier run) already produced them.

In [ ]:
with timed("clean"):
    C.clean_all(DATA_DIR, CLEAN_DIR, jobs=JOBS)

FILES = {(split, src): CLEAN_DIR / f"{split}_source{src[-1]}.parquet"
         for split in ("train", "test") for src in ("S1", "S2", "S3")}
with timed("keys"), ProcessPoolExecutor(max_workers=JOBS) as ex:
    futures = {k: ex.submit(B.cache_keys, p, CACHE_DIR) for k, p in FILES.items()}
    chunks = {k: f.result() for k, f in futures.items()}

query = {s: chunks[(s, "S1")] for s in ("train", "test")}
pool = {s: chunks[(s, "S2")] + chunks[(s, "S3")] for s in ("train", "test")}
with timed("pool stats"):
    stats = {s: B.pool_stats(pool[s]) for s in ("train", "test")}
n_pool = {s: sum(stats[s].n.values()) for s in stats}
pd.DataFrame({s: {**stats[s].n, "S1 records": sum(B.load_chunk(p).n for p in query[s])} for s in stats})

## 3. Train candidates (all 2.2M S1), labelled
Every candidate pair gets `label = 1` if the ground truth links it. True matches that blocking
missed are **not** in this table (they are counted in `blocking_per_s1_train.parquet`), so a matcher
trained here sees exactly the distribution it will face at inference.

In [ ]:
if RUN_TRAIN_CANDIDATES:
    gt = pd.read_csv(DATA_DIR / "train/train_ground_truth.tsv", **C.READ_KW)
    ex_ = gt.assign(cand=gt.matched_entity_ids.str.split(",")).explode("cand").query("cand != ''")
    truth = pd.DataFrame({"s1": B.encode_ids(ex_.source1_entity_id), "cand": B.encode_ids(ex_.cand)})
    del gt, ex_
    with timed("train candidates"):
        per_q_train = B.generate_candidates(query["train"], pool["train"], stats["train"],
                                            ARTIFACT_DIR / "candidates_train", passes=B.FINAL_PASSES,
                                            truth=truth, batch_chunks=BATCH_CHUNKS)
    per_q_train.to_parquet(ARTIFACT_DIR / "blocking_per_s1_train.parquet", index=False)
    train_metrics = B.summarise(per_q_train, n_pool["train"])
    display(pd.Series(train_metrics))

## 4. Test candidates (all 1.7M S1) → `output/candidate_pairs.tsv`
IDF and frequency caps come from the test pool itself; the unseen `France` partition is handled
like any other country.

In [ ]:
if RUN_TEST_CANDIDATES:
    with timed("test candidates"):
        per_q_test = B.generate_candidates(query["test"], pool["test"], stats["test"],
                                           ARTIFACT_DIR / "candidates_test", passes=B.FINAL_PASSES,
                                           tsv_path=OUT_DIR / "candidate_pairs.tsv", batch_chunks=BATCH_CHUNKS)
    per_q_test.to_parquet(ARTIFACT_DIR / "blocking_per_s1_test.parquet", index=False)
    test_metrics = B.summarise(per_q_test, n_pool["test"])
    display(pd.Series(test_metrics))

    test_country = np.concatenate([B.load_chunk(p).country for p in query["test"]])
    display(pd.DataFrame({"country": test_country, "n": per_q_test.n_candidates}).groupby("country").n.describe())

In [ ]:
# Official validator on the candidate file (an all-empty matching file stands in, as a valid subset)
if RUN_TEST_CANDIDATES:
    with tempfile.TemporaryDirectory() as tmp:
        empty = Path(tmp) / "matching_results.tsv"
        pd.DataFrame({"source1_entity_id": B.chunk_ids(query["test"]).astype(str), "matched_entity_ids": ""}) \
            .to_csv(empty, sep="\t", index=False)
        r = subprocess.run([sys.executable, str(DATA_DIR.parent / "utils/validate_submission.py"),
                            "--matching", str(empty), "--candidate", str(OUT_DIR / "candidate_pairs.tsv"),
                            "--test-dir", str(DATA_DIR / "test")], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-500:])
    assert r.returncode == 0, "candidate_pairs.tsv failed validation"

## 5. Report

In [ ]:
report = {
    "passes": [{"name": p.name, "families": p.families, "k": p.k, "cap": p.cap, "min_score": p.min_score}
               for p in B.FINAL_PASSES],
    "key_bits": B.KEY_BITS,
    "pool_size": n_pool,
    "train": locals().get("train_metrics"),
    "test": locals().get("test_metrics"),
    "timings_s": timings,
    "finished": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(ARTIFACT_DIR / "blocking_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## 6. Using the artifacts in the training pipeline

```python
import pandas as pd

cand = pd.read_parquet("artifacts/candidates_train")            # all parts; ~60M rows, so filter columns if needed
s1 = pd.read_parquet("cleaned/train_source1.parquet")
pool = pd.concat([pd.read_parquet(f"cleaned/train_source{i}.parquet") for i in (2, 3)])

pairs = (cand.merge(s1.add_prefix("l_"), left_on="s1_entity_id", right_on="l_entity_id")
             .merge(pool.add_prefix("r_"), left_on="cand_entity_id", right_on="r_entity_id"))
# → features from l_* vs r_* (name/address similarities, legal_form conflict, region match, ...)
#   plus the blocking features score, score_*, rank_*; target = label
```

* Split train/validation **by `s1_entity_id`** (all candidates of an entity on the same side), and
  score validation with the per-entity F0.5 using `blocking_per_s1_train.parquet` so missed true
  matches count against recall.
* Singletons (entities with `n_true == 0`) score 1.0 only if *no* candidate is predicted, so the
  decision threshold should be tuned on the macro F0.5, not on pair accuracy.
* At inference, read `artifacts/candidates_test` with `cleaned/test_source*.parquet`; the final
  `matching_results.tsv` must be a subset of `output/candidate_pairs.tsv`.